# Image Tiler Evaluation

This notebook evaluates `flask-application/image_tiler.py`, the shared tiler used for model preprocessing.

The goal is not to measure model accuracy. The goal is to check whether the tiler prepares images correctly before they reach the mangrove detector, multi-class classifier, or future tree counter.

## What To Evaluate

A good image tiler should pass these checks:

- **Coverage:** every pixel in the original image should be included in at least one tile.
- **Edge handling:** right and bottom edges should not be skipped.
- **Padding:** small images and edge tiles should be padded to the model's expected fixed size.
- **Overlap:** overlapping tiles should be generated consistently using `tile_size - overlap` as the step.
- **Position tracking:** every tile should return its original-image coordinates.
- **Empty tile skipping:** optional blank/near-empty tile filtering should reduce useless tiles.
- **Model compatibility:** transformed tiles should stack into a tensor batch for PyTorch models.
- **Performance:** tile count and runtime should be reasonable for large drone images.

In [ ]:
from pathlib import Path
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageDraw

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FLASK_APP_DIR = PROJECT_ROOT / "flask-application"
sys.path.insert(0, str(FLASK_APP_DIR))

from image_tiler import tile_image, tile_image_for_model

print("Project root:", PROJECT_ROOT)
print("Using tiler:", FLASK_APP_DIR / "image_tiler.py")

## Helper Functions

In [ ]:
def coverage_map(image_size, positions):
    width, height = image_size
    coverage = np.zeros((height, width), dtype=np.uint16)
    for left, top, right, bottom in positions:
        coverage[top:bottom, left:right] += 1
    return coverage


def summarize_tiling(image_size, tile_size, overlap, skip_empty=False):
    image = Image.new("RGB", image_size, (40, 90, 60))
    start = time.perf_counter()
    tiles, positions, original_size = tile_image(
        image,
        tile_size=tile_size,
        overlap=overlap,
        skip_empty=skip_empty,
    )
    elapsed = time.perf_counter() - start
    coverage = coverage_map(original_size, positions)
    summary = {
        "image_size": original_size,
        "tile_size": tile_size,
        "overlap": overlap,
        "tile_count": len(tiles),
        "all_tiles_fixed_size": all(tile.size == (tile_size, tile_size) for tile in tiles),
        "uncovered_pixels": int((coverage == 0).sum()),
        "min_coverage": int(coverage.min()) if coverage.size else 0,
        "max_coverage": int(coverage.max()) if coverage.size else 0,
        "first_position": positions[0] if positions else None,
        "last_position": positions[-1] if positions else None,
        "runtime_seconds": round(elapsed, 4),
    }
    return summary, tiles, positions, coverage


def draw_tile_grid(image_size, positions, title):
    image = Image.new("RGB", image_size, (40, 90, 60))
    draw = ImageDraw.Draw(image)
    for index, box in enumerate(positions):
        draw.rectangle(box, outline=(255, 230, 80), width=3)
        draw.text((box[0] + 6, box[1] + 6), str(index), fill=(255, 255, 255))
    plt.figure(figsize=(7, 5))
    plt.imshow(image)
    plt.title(title)
    plt.axis("off")
    plt.show()

## Test 1: Standard Tiling Cases

This checks small images, exact-size images, larger images, and tree-counter-style `640x640` tiling.

In [ ]:
test_cases = [
    {"image_size": (300, 300), "tile_size": 512, "overlap": 64, "name": "small image"},
    {"image_size": (512, 512), "tile_size": 512, "overlap": 64, "name": "exact 512 image"},
    {"image_size": (900, 700), "tile_size": 512, "overlap": 64, "name": "large classifier image"},
    {"image_size": (1300, 900), "tile_size": 640, "overlap": 64, "name": "large tree-counter image"},
]

summaries = []
for case in test_cases:
    summary, tiles, positions, coverage = summarize_tiling(
        case["image_size"],
        case["tile_size"],
        case["overlap"],
    )
    summary["case"] = case["name"]
    summaries.append(summary)

    assert summary["tile_count"] >= 1
    assert summary["all_tiles_fixed_size"] is True
    assert summary["uncovered_pixels"] == 0
    assert summary["last_position"][2] == case["image_size"][0]
    assert summary["last_position"][3] == case["image_size"][1]

summaries

## Test 2: Visual Coverage Check

The grid should cover the whole image, including the right and bottom edges.

In [ ]:
case = {"image_size": (900, 700), "tile_size": 512, "overlap": 64}
summary, tiles, positions, coverage = summarize_tiling(**case)
draw_tile_grid(case["image_size"], positions, "512x512 tiles with 64px overlap")

plt.figure(figsize=(7, 5))
plt.imshow(coverage, cmap="viridis")
plt.colorbar(label="coverage count")
plt.title("Coverage map: every pixel should be >= 1")
plt.axis("off")
plt.show()

summary

## Test 3: Empty Tile Skipping

This creates a mostly black image with one content area. With `skip_empty=True`, the tiler should keep fewer tiles.

In [ ]:
img = Image.new("RGB", (1200, 800), (0, 0, 0))
draw = ImageDraw.Draw(img)
draw.rectangle((150, 120, 700, 620), fill=(30, 120, 65))

all_tiles, all_positions, _ = tile_image(img, tile_size=512, overlap=64, skip_empty=False)
kept_tiles, kept_positions, _ = tile_image(
    img,
    tile_size=512,
    overlap=64,
    skip_empty=True,
    min_content_fraction=0.01,
    blank_threshold=5,
)

print("Tiles without empty skipping:", len(all_tiles))
print("Tiles with empty skipping:", len(kept_tiles))
print("Skipped tiles:", len(all_tiles) - len(kept_tiles))

draw_tile_grid(img.size, kept_positions, "Tiles kept after empty-tile filtering")

## Test 4: PyTorch Model Input Shape

`tile_image_for_model` should return a stacked tensor batch suitable for the existing PyTorch models.

In [ ]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

img = Image.new("RGB", (900, 700), (40, 90, 60))
batch, positions = tile_image_for_model(img, transform, tile_size=512, overlap=64)

print("Batch shape:", tuple(batch.shape))
print("Number of positions:", len(positions))

assert batch.ndim == 4
assert batch.shape[0] == len(positions)
assert batch.shape[1:] == (3, 224, 224)

## Optional: Evaluate A Real Drone Image

Set `REAL_IMAGE_PATH` to a local drone image path. This will report tile count, coverage, and runtime.

In [ ]:
REAL_IMAGE_PATH = None
# Example:
# REAL_IMAGE_PATH = PROJECT_ROOT / "flask-application" / "uploads" / "example.jpg"

if REAL_IMAGE_PATH is not None:
    real_image = Image.open(REAL_IMAGE_PATH).convert("RGB")
    summary, tiles, positions, coverage = summarize_tiling(real_image.size, tile_size=512, overlap=64)
    print(summary)
    draw_tile_grid(real_image.size, positions, f"Real image tiling: {REAL_IMAGE_PATH.name}")
else:
    print("Set REAL_IMAGE_PATH to evaluate a real drone image.")

## Suggested Evaluation Summary

When documenting the image tiler, report:

- Tile size used for each model: `512x512` for the current mangrove models, `640x640` for the future tree counter.
- Number of tiles produced for sample small, medium, and large images.
- Whether uncovered pixels equal `0`.
- Whether every output tile has the expected fixed size.
- Whether the final tile reaches the original image's right and bottom edges.
- Runtime for a representative drone image.
- Empty-tile skip rate, if `skip_empty=True` is used.

For model evaluation, keep it separate: after confirming the tiler works, evaluate whether model predictions stay consistent or improve when using the shared tiler.